# Export Fine-Tuned Gemma 2 → GGUF for Ollama (Colab — No Drive)

### Before you start:
1. **Runtime → Change runtime type → T4 GPU**
2. Run cells top to bottom
3. When prompted, upload your adapter files from `fine tuned model/mi-therapy-gemma2-v2/`

## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers>=4.44.0 peft>=0.12.0 accelerate>=0.33.0 bitsandbytes
!pip install -q gguf sentencepiece protobuf

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    raise RuntimeError('No GPU! Go to Runtime → Change runtime type → T4 GPU')
print('\n✅ Ready')

## Step 2: Upload Your Adapter Files

A file picker will pop up. Select **ALL 6 files** from `fine tuned model/mi-therapy-gemma2-v2/`:
- `adapter_config.json`
- `adapter_model.safetensors`
- `chat_template.jinja`
- `README.md`
- `tokenizer.json`
- `tokenizer_config.json`

In [ ]:
import os
from google.colab import files

ADAPTER_PATH = '/content/adapter'
os.makedirs(ADAPTER_PATH, exist_ok=True)

print('Select ALL 6 adapter files when the file picker opens...\n')
uploaded = files.upload()

for filename, data in uploaded.items():
    filepath = os.path.join(ADAPTER_PATH, filename)
    with open(filepath, 'wb') as f:
        f.write(data)

print(f'\n✅ Uploaded {len(uploaded)} files to {ADAPTER_PATH}:')
for f in sorted(os.listdir(ADAPTER_PATH)):
    size = os.path.getsize(os.path.join(ADAPTER_PATH, f))
    label = f'{size/1024:.1f} KB' if size < 1024*1024 else f'{size/1024/1024:.1f} MB'
    print(f'   {f} ({label})')

assert 'adapter_config.json' in os.listdir(ADAPTER_PATH), '❌ adapter_config.json missing!'
assert 'adapter_model.safetensors' in os.listdir(ADAPTER_PATH), '❌ adapter_model.safetensors missing!'
print('\n✅ Adapter files verified!')

## Step 3: Load Base Model + Merge LoRA Adapter

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL = 'google/gemma-2-2b-it'
MERGED_PATH = '/content/gemma2-mi-merged'

print('Loading base model (1-2 minutes)...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='auto'
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print(f'Loading LoRA adapter from {ADAPTER_PATH}...')
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

print('Merging LoRA weights into base model...')
merged_model = model.merge_and_unload()

print(f'Saving merged model to {MERGED_PATH}...')
merged_model.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)

print(f'\n✅ Merge complete!')
print(f'   GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB')

## Step 4: Quick Sanity Test

In [ ]:
from transformers import pipeline

pipe = pipeline('text-generation', model=merged_model, tokenizer=tokenizer, max_new_tokens=150)

test_messages = [
    {'role': 'user', 'content': "I don't think I have a problem with drinking. Everyone does it."}
]

prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
output = pipe(prompt, do_sample=True, temperature=0.7, top_p=0.9)

print('Prompt: "I don\'t think I have a problem with drinking. Everyone does it."')
print(f'\nModel response:')
print(output[0]['generated_text'][len(prompt):])
print('\n✅ If it sounds like an MI therapist, the merge worked!')

## Step 5: Convert to GGUF

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
print('\n✅ llama.cpp ready')

In [ ]:
!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/gemma2-mi-merged \
    --outfile /content/gemma2-mi-therapist.gguf \
    --outtype q4_K_M

import os
gguf_path = '/content/gemma2-mi-therapist.gguf'
size_gb = os.path.getsize(gguf_path) / (1024**3)
print(f'\n✅ GGUF created: {size_gb:.2f} GB')

## Step 6: Download GGUF to Your PC

In [ ]:
from google.colab import files

print('Starting download of gemma2-mi-therapist.gguf...')
print('(~1.5 GB — may take a few minutes)\n')
files.download('/content/gemma2-mi-therapist.gguf')
print('\n✅ Download started! Check your browser downloads.')

## Done! Next Steps (on your laptop)

### 1. Install Ollama
Download from https://ollama.com

### 2. Create Modelfile
In the same folder as the downloaded GGUF, create a file called `Modelfile`:
```
FROM ./gemma2-mi-therapist.gguf

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.2
PARAMETER num_predict 200

TEMPLATE """<start_of_turn>user
{{ .Prompt }}<end_of_turn>
<start_of_turn>model
{{ .Response }}<end_of_turn>
"""
```

### 3. Create Ollama model
```bash
ollama create mi-therapist -f Modelfile
```

### 4. Test
```bash
ollama run mi-therapist "I don't think I have a problem with drinking."
```

### 5. Update chatbot .env
```
OLLAMA_MODEL=mi-therapist
```

Then `streamlit run app.py` — your chatbot uses YOUR model!